# 视频字幕生成
本笔记本展示了如何使用 VideoCaptioningChain，该链通过 Langchain 的 ImageCaptionLoader 和 AssemblyAI 实现，用于生成 .srt 文件。

该系统可自动为视频 URL 生成字幕和隐藏式字幕。

## 安装依赖项

In [1]:
# !pip install ffmpeg-python
# !pip install assemblyai
# !pip install opencv-python
# !pip install torch
# !pip install pillow
# !pip install transformers
# !pip install langchain

## 导入

In [1]:
import getpass

from langchain.chains.video_captioning import VideoCaptioningChain
from langchain.chat_models.openai import ChatOpenAI

## 设置 API 密钥

API 密钥是用于身份验证的字符串。它们允许您访问 API。

### 注册以获取 API 密钥

1.  访问 [OpenAI 网站](https://platform.openai.com/)。
2.  如果您还没有帐户，请注册一个帐户。
3.  登录后，导航到“API 密钥”页面。您可以在此处找到您的 API 密钥。

### 保护您的 API 密钥

API 密钥是敏感信息，应像密码一样进行保护。

*   **切勿**将您的 API 密钥公开在客户端代码（例如 JavaScript）中。
*   **切勿**将您的 API 密钥提交到公共代码存储库（例如 GitHub）。
*   考虑使用环境变量来存储您的 API 密钥。

### 使用 API 密钥

您可以通过在请求的 `Authorization` 标头中包含您的 API 密钥来使用它：

```
Authorization: Bearer YOUR_API_KEY
```

将 `YOUR_API_KEY` 替换为您实际的 API 密钥。

In [2]:
OPENAI_API_KEY = getpass.getpass("OpenAI API Key:")

ASSEMBLYAI_API_KEY = getpass.getpass("AssemblyAI API Key:")

**必需参数：**

*   llm：此链将使用的语言模型，用于获取有关如何优化闭路字幕的建议
*   assemblyai\_key：AssemblyAI 的 API 密钥，用于生成字幕

**可选参数：**

*   verbose（默认值：True）：为下游链调用设置详细模式
*   use\_logging（默认值：True）：在运行管理器中记录链的处理过程
*   frame\_skip（默认值：None）：选择在处理过程中跳过多少视频帧。增加此值会加快执行速度，但结果会降低准确性。如果为 None，则帧跳过是根据帧率手动计算的。将其设置为 0 以采样所有帧
*   image\_delta\_threshold（默认值：3000000）：设置图像处理器认为视频中场景变化敏感度，用于界定闭路字幕。值越高 = 敏感度越低
*   closed\_caption\_char\_limit（默认值：20）：设置闭路字幕的字符限制
*   closed\_caption\_similarity\_threshold（默认值：80）：设置两个闭路字幕模型应有多相似才能聚类成一个更长的闭路字幕的百分比值
*   use\_unclustered\_video\_models（默认值：False）：如果为 True，将包含无法聚类的闭路字幕。可能导致闭路字幕出现自发行为，例如持续时间很短或变化很快的字幕。启用此选项是实验性的，不推荐

## 示例运行

This example shows how to use the `useQuery` hook to fetch data from an API.
这个示例展示了如何使用 `useQuery` hook 从 API 获取数据。

```jsx
import { useQuery } from '@tanstack/react-query';
import axios from 'axios';

function Todos() {
  const { isLoading, error, data } = useQuery({
    queryKey: ['todos'],
    queryFn: async () => {
      const { data } = await axios.get('https://jsonplaceholder.typicode.com/todos');
      return data;
    },
  });

  if (isLoading) return 'Loading...';

  if (error) return 'An error has occurred: ' + error.message;

  return (
    <ul>
      {data.map((todo) => (
        <li key={todo.id}>{todo.title}</li>
      ))}
    </ul>
  );
}

In [ ]:
# https://ia804703.us.archive.org/27/items/uh-oh-here-we-go-again/Uh-Oh%2C%20Here%20we%20go%20again.mp4
# https://ia601200.us.archive.org/9/items/f58703d4-61e6-4f8f-8c08-b42c7e16f7cb/f58703d4-61e6-4f8f-8c08-b42c7e16f7cb.mp4

chain = VideoCaptioningChain(
    llm=ChatOpenAI(model="gpt-4", max_tokens=4000, openai_api_key=OPENAI_API_KEY),
    assemblyai_key=ASSEMBLYAI_API_KEY,
)

srt_content = chain.run(
    video_file_path="https://ia601200.us.archive.org/9/items/f58703d4-61e6-4f8f-8c08-b42c7e16f7cb/f58703d4-61e6-4f8f-8c08-b42c7e16f7cb.mp4"
)

print(srt_content)

## 将输出写入 .srt 文件

This guide explains how to write output to a `.srt` file.

本指南介绍如何将输出写入 `.srt` 文件。

`.srt` files are a common format for storing subtitle data. They are plain text files with a specific structure.

`.srt` 文件是存储字幕数据的常用格式。它们是具有特定结构的纯文本文件。

Here's an example of a `.srt` file:

以下是一个 `.srt` 文件的示例：

```srt
1
00:00:01,000 --> 00:00:05,000
This is the first subtitle.

2
00:00:06,000 --> 00:00:10,000
This is the second subtitle.
```

Each subtitle entry consists of:

每个字幕条目包含：

1.  **A sequential number:** This indicates the order of the subtitle.
    1.  **顺序编号：** 这表示字幕的顺序。
2.  **A timecode:** This specifies the start and end time for the subtitle to be displayed. The format is `HH:MM:SS,ms --> HH:MM:SS,ms`.
    2.  **时间码：** 这指定了字幕显示的开始和结束时间。格式为 `HH:MM:SS,ms --> HH:MM:SS,ms`。
3.  **The subtitle text:** This is the actual text that will be displayed.
    3.  **字幕文本：** 这是将要显示的实际文本。

You can create `.srt` files programmatically using various programming languages. The general approach involves:

您可以使用各种编程语言以编程方式创建 `.srt` 文件。通用方法包括：

1.  **Collecting your subtitle data:** This might come from a translation process, speech-to-text output, or manual creation.
    1.  **收集字幕数据：** 这可能来自翻译过程、语音转文本输出或手动创建。
2.  **Formatting the data:** Ensure each subtitle entry follows the `.srt` format (number, timecode, text).
    2.  **格式化数据：** 确保每个字幕条目都遵循 `.srt` 格式（编号、时间码、文本）。
3.  **Writing to a file:** Open a file with a `.srt` extension and write the formatted data to it.

    3.  **写入文件：** 打开一个扩展名为 `.srt` 的文件，并将格式化后的数据写入其中。

Here's a conceptual example using Python:

以下是一个使用 Python 的概念性示例：

```python
def write_srt_file(subtitle_data, filename):
    """
    Writes subtitle data to a .srt file.

    Args:
        subtitle_data: A list of dictionaries, where each dictionary
                       represents a subtitle entry with 'number', 'start_time',
                       'end_time', and 'text' keys.
        filename: The name of the .srt file to create.
    """
    with open(filename, 'w', encoding='utf-8') as f:
        for entry in subtitle_data:
            f.write(f"{entry['number']}\n")
            f.write(f"{entry['start_time']} --> {entry['end_time']}\n")
            f.write(f"{entry['text']}\n\n")

# Example usage:
# subtitle_entries = [
#     {"number": 1, "start_time": "00:00:01,000", "end_time": "00:00:05,000", "text": "Hello, world!"},
#     {"number": 2, "start_time": "00:00:06,000", "end_time": "00:00:10,000", "text": "This is a test."},
# ]
# write_srt_file(subtitle_entries, "output.srt")
```

**Important considerations:**

**重要注意事项：**

*   **Encoding:** Always use UTF-8 encoding when writing `.srt` files to ensure proper display of various characters, especially in different languages.
*   **编码：** 写入 `.srt` 文件时，请务必使用 UTF-8 编码，以确保正确显示各种字符，尤其是在不同语言中。
*   **Timecode Format:** Be precise with the timecode format (`HH:MM:SS,ms`). Incorrect formatting can lead to subtitles not appearing or being out of sync.
*   **时间码格式：** 请精确注意时间码格式（`HH:MM:SS,ms`）。格式不正确可能导致字幕不显示或不同步。
*   **Blank Lines:** Ensure there is a blank line between each subtitle entry. This is crucial for the `.srt` parser to correctly identify separate entries.
*   **空行：** 确保每个字幕条目之间都有一个空行。这对于 `.srt` 解析器正确识别单独的条目至关重要。

In [6]:
with open("output.srt", "w") as file:
    file.write(srt_content)